# 🛒 Customer Churn Prediction – E-Commerce Platform

## STAT3013 – Information System Analysis & Design Project

**Pipeline đầu-đến-cuối (SOTA 2024-2025):**

`EDA → Xử lý Outlier → Feature Engineering → Thống kê suy diễn → Hồi quy CLV → SEM → ML (LightGBM + CatBoost) → DL (FT-Transformer + TabNet) → SHAP → Xuất Dashboard`

### ✅ Các mô hình sử dụng (hiện đại 2024-2025):
| # | Mô hình | Loại | Năm | Nguồn |
|---|---------|------|-----|-------|
| ML1 | **LightGBM** | Gradient Boosting | Microsoft 2017 | Ke et al., NeurIPS 2017 |
| ML2 | **CatBoost** | Gradient Boosting | Yandex 2018 | Prokhorenkova et al., NeurIPS 2018 |
| DL1 | **FT-Transformer** | Transformer cho dữ liệu bảng | Yandex 2022 | Gorishniy et al., NeurIPS 2021 |
| DL2 | **TabNet** | DL dựa trên Attention | Google 2021 | Arik & Pfister, AAAI 2021 |

### 📋 Cấu trúc notebook:
1. Cài đặt thư viện  2. Import  3. Tải dữ liệu  4. EDA  5. Winsorization
6. Feature Engineering  7. Thống kê suy diễn  8. Hồi quy CLV  9. SEM
10. Pipeline tiền xử lý ML/DL  11. LightGBM+SHAP  12. CatBoost
13. FT-Transformer  14. TabNet  15. So sánh mô hình  16. Xuất CSV  17. Kết luận

## 📦 0. Cài đặt thư viện (chạy một lần)

> **Lưu ý Colab:** Sau khi cài, restart runtime nếu được nhắc, rồi chạy lại từ đầu.

In [ ]:
# Cài đặt / nâng cấp pip và các thư viện cần thiết (chỉ chạy lần đầu)
!pip install --upgrade pip --quiet                          # Nâng cấp pip lên bản mới nhất
!pip install "numpy>=1.22.4,<2.0.0" --quiet                # NumPy (giới hạn <2.0 để tránh lỗi tương thích)
!pip install lightgbm==4.3.0 catboost==1.2.5 shap==0.45.0 --quiet  # Các model ML chính + SHAP giải thích
!pip install imbalanced-learn==0.12.3 --quiet              # SMOTE xử lý mất cân bằng nhãn
!pip install pytorch-tabnet==4.1.0 --quiet                 # TabNet – DL cho dữ liệu bảng
!pip install semopy==2.3.11 --quiet                        # SEM (Structural Equation Modeling)
!pip install openpyxl --quiet                              # Đọc/ghi file Excel .xlsx
print("✅ Cài đặt hoàn tất! Nếu Colab yêu cầu, restart runtime rồi chạy lại từ đầu.")

## 📚 1. Import thư viện

In [ ]:
# ─── Xử lý dữ liệu ───
import pandas as pd                          # Thao tác DataFrame (đọc file, lọc, nhóm, ...)
import numpy as np                           # Tính toán số học, mảng đa chiều
import matplotlib.pyplot as plt              # Vẽ biểu đồ cơ bản
import seaborn as sns                        # Vẽ biểu đồ thống kê đẹp hơn (boxplot, heatmap, ...)
from scipy import stats                      # Thống kê tổng quát (scipy)
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu
# chi2_contingency: kiểm định Chi-square độc lập
# ttest_ind: kiểm định t hai mẫu độc lập (Welch's t-test)
# mannwhitneyu: kiểm định phi tham số Mann-Whitney U (backup)

# ─── Tiền xử lý ───
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
# train_test_split: chia tập train/test
# StratifiedKFold: k-fold giữ nguyên tỉ lệ nhãn trong mỗi fold
# cross_val_score: đánh giá mô hình qua nhiều fold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
# StandardScaler: chuẩn hoá (mean=0, std=1) – dùng cho DL
# MinMaxScaler: scale về [0,1] – dùng cho SEM
# LabelEncoder: mã hoá nhãn dạng số (không dùng trực tiếp nhưng import sẵn)
from sklearn.impute import KNNImputer        # Điền giá trị missing bằng K láng giềng gần nhất
from imblearn.over_sampling import SMOTE     # Tạo thêm mẫu tổng hợp cho lớp thiểu số

# ─── Mô hình ML ───
from lightgbm import LGBMClassifier          # LightGBM – Gradient Boosting nhanh (Microsoft, NeurIPS 2017)
from catboost import CatBoostClassifier      # CatBoost – xử lý tốt biến phân loại (Yandex, NeurIPS 2018)

# ─── Hồi quy CLV ───
from sklearn.ensemble import GradientBoostingRegressor   # Hồi quy Gradient Boosting (dự đoán CLV)
from sklearn.metrics import r2_score, mean_squared_error # R² và RMSE để đánh giá hồi quy

# ─── Metrics phân loại ───
from sklearn.metrics import (
    accuracy_score,            # Tỉ lệ dự đoán đúng
    f1_score,                  # F1 = harmonic mean(Precision, Recall) – quan trọng khi mất cân bằng
    roc_auc_score,             # AUC của đường cong ROC
    precision_score,           # Precision = TP/(TP+FP)
    recall_score,              # Recall = TP/(TP+FN)
    classification_report,     # Báo cáo chi tiết Precision/Recall/F1 theo từng lớp
    confusion_matrix,          # Ma trận nhầm lẫn (TP, FP, TN, FN)
    roc_curve,                 # Tọa độ đường cong ROC
    auc,                       # Diện tích dưới đường cong (dùng cho PR-AUC)
    average_precision_score,   # PR-AUC – quan trọng khi dữ liệu mất cân bằng
    precision_recall_curve     # Tọa độ đường cong Precision-Recall
)

# ─── SHAP – Giải thích mô hình ───
import shap                    # Tính SHAP values để hiểu tại sao mô hình đưa ra dự đoán đó

# ─── Deep Learning ───
import torch                              # Framework PyTorch
import torch.nn as nn                     # Các lớp neural network (Linear, LayerNorm, ...)
import torch.optim as optim               # Các optimizer (AdamW, ...)
from torch.utils.data import DataLoader, TensorDataset
# DataLoader: chia batch, shuffle data khi training
# TensorDataset: bọc tensor X và y thành một dataset
from pytorch_tabnet.tab_model import TabNetClassifier  # TabNet (AAAI 2021) – DL có attention

# ─── SEM – Mô hình phương trình cấu trúc ───
try:
    import semopy              # semopy: thư viện SEM trong Python
    SEM_AVAILABLE = True
    print("✅ semopy đã tải (SEM đầy đủ sẽ chạy)")
except ImportError:
    SEM_AVAILABLE = False      # Nếu chưa cài, chỉ vẽ sơ đồ đường dẫn
    print("⚠️  semopy chưa có – chỉ vẽ path diagram")

import warnings
warnings.filterwarnings("ignore")   # Ẩn các cảnh báo không cần thiết khi chạy

# ─── Cài đặt toàn cục ───
plt.style.use('seaborn-v0_8-whitegrid')    # Phong cách biểu đồ sáng, có lưới
plt.rcParams['figure.facecolor'] = 'white' # Nền trắng cho tất cả biểu đồ
SEED = 42                                  # Hạt giống ngẫu nhiên – đảm bảo tái lập kết quả
np.random.seed(SEED)                       # Fix seed cho NumPy
torch.manual_seed(SEED)                    # Fix seed cho PyTorch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Dùng GPU nếu có, không thì CPU
print(f"✅ Import hoàn tất")
print(f"   • PyTorch: {torch.__version__}")
print(f"   • Thiết bị: {DEVICE}")
print(f"   • Seed:     {SEED}")

In [ ]:
import torch

# Kiểm tra GPU có sẵn hay không
if torch.cuda.is_available():
    print("✅ GPU khả dụng!")
    print(f"   Tên GPU: {torch.cuda.get_device_name(0)}")  # In tên card đồ họa
    DEVICE = 'cuda'   # Đặt thiết bị là GPU
else:
    print("❌ Không có GPU. Chạy trên CPU.")
    DEVICE = 'cpu'    # Đặt thiết bị là CPU

## 📂 2. Tải dữ liệu

In [ ]:
# Đường dẫn file Excel – kéo thả file vào Colab hoặc chạy trên cùng thư mục
FILE_PATH = 'DataPttk_E_commerce.xlsx'

df = pd.read_excel(FILE_PATH)    # Đọc toàn bộ file Excel vào DataFrame df

# In tổng quan: số hàng, cột, tỉ lệ churn
print(f"📊 Kích thước:   {df.shape[0]:,} hàng × {df.shape[1]} cột")
print(f"🎯 Tỉ lệ churn:  {df['Churn'].mean()*100:.2f}%  "
      f"({df['Churn'].sum()} churn / {(df['Churn']==0).sum()} không churn)")
print(f"\n📋 Danh sách cột:")
for i, col in enumerate(df.columns, 1):        # Duyệt từng cột và in số thứ tự + tên
    print(f"   {i:2d}. {col}")

df.head()    # Hiện 5 hàng đầu để xem sơ bộ

## 🔍 3. Khám phá dữ liệu (EDA)

### 3.1 Tổng quan

In [ ]:
print("=" * 60)
print("TỔNG QUAN DỮ LIỆU")
print("=" * 60)
df.info()                       # In kiểu dữ liệu, số non-null từng cột
print("\n--- Thống kê mô tả (đặc trưng số) ---")
df.describe().round(2)          # Tóm tắt min/max/mean/std/phân vị cho cột số

### 3.2 Phân tích giá trị thiếu (Missing Values)

In [ ]:
missing = df.isnull().sum()                         # Đếm số NaN mỗi cột
missing_pct = (missing / len(df) * 100).round(2)   # Tính phần trăm thiếu mỗi cột

# Tạo bảng tóm tắt chỉ gồm các cột có giá trị thiếu, sắp xếp giảm dần
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage (%)': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(
    'Missing Count', ascending=False)

print("🔍 GIÁ TRỊ THIẾU:")
print(missing_df)
print(f"\n📊 Tổng thiếu: {missing.sum():,} / {df.size:,} ({missing.sum()/df.size*100:.2f}%)")
print("💡 Kết luận: ≤ 5.5% mỗi cột → KNN Imputer sẽ xử lý.")

# Vẽ biểu đồ thanh ngang nếu có missing
if len(missing_df) > 0:
    plt.figure(figsize=(10, 4))
    missing_df['Percentage (%)'].plot(kind='barh', color='coral', edgecolor='black')
    plt.title('Phần trăm giá trị thiếu theo cột', fontsize=13, fontweight='bold')
    plt.xlabel('Thiếu (%)')
    plt.tight_layout()
    plt.show()

### 3.3 Phân phối Churn – Kiểm tra mất cân bằng nhãn

In [ ]:
churn_counts = df['Churn'].value_counts()   # Đếm số khách hàng churn (1) và không churn (0)
churn_rate = df['Churn'].mean() * 100        # Tỉ lệ churn (%)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))   # Tạo 3 ô vẽ nằm ngang

# --- Ô 1: Biểu đồ cột đếm số lượng ---
sns.countplot(x='Churn', data=df, ax=axes[0],
              palette=['#2196F3', '#F44336'], order=[0, 1])
axes[0].set_title('Phân phối Churn (Số lượng)', fontweight='bold')
axes[0].set_xticklabels(['Không Churn\n(0)', 'Churn\n(1)'])
for p in axes[0].patches:   # Ghi số lượng lên đầu mỗi cột
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

# --- Ô 2: Biểu đồ tròn tỉ lệ ---
axes[1].pie(churn_counts, labels=['Không Churn', 'Churn'],
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'],
            startangle=90, explode=(0, 0.05))   # explode làm nổi bật lớp Churn
axes[1].set_title('Tỉ lệ Churn', fontweight='bold')

# --- Ô 3: Cảnh báo và giải pháp ---
axes[2].axis('off')   # Tắt trục, chỉ hiện text
axes[2].text(0.5, 0.7, '⚠️ DỮ LIỆU MẤT CÂN BẰNG', ha='center', fontsize=14,
             fontweight='bold', color='red', transform=axes[2].transAxes)
axes[2].text(0.5, 0.5, f'Tỉ lệ Churn: {churn_rate:.2f}%', ha='center',
             fontsize=16, fontweight='bold', color='darkred',
             transform=axes[2].transAxes)
axes[2].text(0.5, 0.25,
             'Giải pháp:\n• SMOTE oversampling\n• Dùng F1/AUC (KHÔNG dùng accuracy!)',
             ha='center', fontsize=11, color='navy', transform=axes[2].transAxes)

plt.suptitle('Biến Mục Tiêu – Phân Phối Churn', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# In tỉ lệ mất cân bằng
print(f"\n📊 Tỉ lệ mất cân bằng: {(1-df['Churn'].mean())/df['Churn'].mean():.2f} : 1 "
      f"(Không Churn : Churn)")

### 3.4 Phân tích Outlier (phương pháp IQR)

In [ ]:
print("=" * 60)
print("PHÂN TÍCH OUTLIER – PHƯƠNG PHÁP IQR")
print("=" * 60)

# Danh sách các cột số cần kiểm tra outlier
numeric_cols = ['Tenure', 'WarehouseToHome', 'HourSpendOnApp',
                'NumberOfDeviceRegistered', 'SatisfactionScore',
                'NumberOfAddress', 'CouponUsed', 'OrderCount',
                'DaySinceLastOrder', 'CashbackAmount', 'OrderAmountHikeFromlastYear']

outlier_report = []
for col in numeric_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)          # Tứ phân vị thứ nhất (25%)
        Q3 = df[col].quantile(0.75)          # Tứ phân vị thứ ba (75%)
        IQR = Q3 - Q1                        # Khoảng tứ phân vị (Interquartile Range)
        lower = Q1 - 1.5 * IQR              # Ngưỡng dưới: thấp hơn là outlier
        upper = Q3 + 1.5 * IQR              # Ngưỡng trên: cao hơn là outlier
        n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()  # Đếm số outlier
        pct = n_outliers / df[col].notna().sum() * 100               # % outlier trên tổng non-null
        outlier_report.append({
            'Feature': col,
            'Lower': round(lower, 2),
            'Upper': round(upper, 2),
            'N_Outliers': n_outliers,
            'Pct(%)': round(pct, 2)
        })

outlier_df = pd.DataFrame(outlier_report).sort_values('N_Outliers', ascending=False)
print(outlier_df.to_string(index=False))

print("\n💡 Quyết định xử lý:")
print("   • KHÔNG xóa outlier (~3-5% dữ liệu, mang ý nghĩa kinh doanh thực)")
print("   • Áp dụng Winsorization (giới hạn tại phân vị 1% / 99%) ở bước tiền xử lý")

In [ ]:
# Vẽ boxplot từng cột số, tô màu theo nhóm Churn
fig, axes = plt.subplots(3, 4, figsize=(20, 14))   # Lưới 3 hàng × 4 cột
axes = axes.flatten()   # Chuyển mảng 2D thành 1D để dễ lặp

for i, col in enumerate(numeric_cols):
    if col in df.columns:
        sns.boxplot(x='Churn', y=col, data=df, ax=axes[i],
                    palette=['#2196F3', '#F44336'])   # Xanh = không churn, đỏ = churn
        axes[i].set_title(f'{col}', fontweight='bold')
        axes[i].set_xticklabels(['Không Churn', 'Churn'])

# Ẩn các ô thừa (nếu số cột < số ô)
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Đặc trưng số – Phân phối theo Churn (thấy rõ outlier)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Tỉ lệ Churn theo đặc trưng phân loại

In [ ]:
# Danh sách cột phân loại cần phân tích
cat_cols = ['PreferredLoginDevice', 'CityTier', 'PreferredPaymentMode',
            'Gender', 'PreferedOrderCat', 'MaritalStatus', 'Complain']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))   # Lưới 2×4
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    if col in df.columns:
        # Tính tỉ lệ churn trung bình theo từng giá trị của cột phân loại
        churn_rate_cat = df.groupby(col)['Churn'].mean() * 100
        churn_rate_cat.sort_values(ascending=False).plot(
            kind='bar', ax=axes[i], color='coral', edgecolor='black')
        axes[i].set_title(f'Tỉ lệ Churn (%) theo {col}', fontweight='bold')
        axes[i].set_ylabel('Tỉ lệ Churn (%)')
        axes[i].tick_params(axis='x', rotation=30)
        # Vẽ đường tỉ lệ churn tổng thể làm tham chiếu
        axes[i].axhline(df['Churn'].mean()*100, ls='--', color='red',
                        label=f'Tổng = {df["Churn"].mean()*100:.1f}%')
        axes[i].legend(fontsize=8)
        for p in axes[i].patches:   # Ghi phần trăm lên mỗi cột
            axes[i].annotate(f'{p.get_height():.1f}%',
                             (p.get_x() + p.get_width()/2., p.get_height()),
                             ha='center', va='bottom', fontsize=8)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Tỉ lệ Churn theo Đặc trưng Phân loại', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6 Heatmap tương quan

In [ ]:
# Chọn chỉ các cột số, bỏ CustomerID (không có ý nghĩa thống kê)
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['CustomerID'])

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(numeric_df.corr(), dtype=bool))   # Che tam giác trên để tránh trùng
sns.heatmap(numeric_df.corr(),        # Ma trận tương quan Pearson
            annot=True,               # Hiện giá trị tương quan trong ô
            fmt='.2f',                # Định dạng 2 chữ số thập phân
            cmap='RdYlGn',            # Màu: đỏ (âm) → vàng (0) → xanh (dương)
            linewidths=0.5,           # Đường kẻ mỏng giữa các ô
            mask=mask,                # Ẩn tam giác trên
            center=0,                 # Đặt 0 làm trung tâm màu
            square=True,              # Ô vuông
            cbar_kws={'label': 'Tương quan Pearson'})
plt.title('Heatmap Tương quan – Đặc trưng số', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# In top 10 đặc trưng tương quan mạnh nhất với Churn
corr_with_churn = numeric_df.corr()['Churn'].abs().sort_values(ascending=False)
print("\n🎯 Top 10 đặc trưng tương quan nhất với Churn:")
print(corr_with_churn.head(11).iloc[1:])   # iloc[1:] bỏ 'Churn' tự tương quan với chính nó

## 🔧 4. Xử lý Outlier – Winsorization

**Tại sao dùng Winsorization thay vì xóa outlier?**
- Giữ nguyên 5.630 hàng (không mất dữ liệu)
- Giới hạn giá trị cực đoan tại phân vị 1% / 99% → giảm ảnh hưởng nhiễu
- Phù hợp thực tế (khách ở xa kho vẫn là khách hàng thật, không phải lỗi nhập liệu)

In [ ]:
print("=" * 60)
print("WINSORIZATION – GIỚI HẠN GIÁ TRỊ CỰC ĐOAN")
print("=" * 60)

df_clean = df.copy()   # Tạo bản sao để không làm thay đổi df gốc

# Các cột có nhiều outlier và ảnh hưởng đến model
cols_to_winsorize = ['WarehouseToHome', 'OrderCount', 'CouponUsed',
                      'NumberOfAddress', 'HourSpendOnApp']

for col in cols_to_winsorize:
    if col in df_clean.columns:
        Q1  = df_clean[col].quantile(0.01)   # Ngưỡng dưới: phân vị 1%
        Q99 = df_clean[col].quantile(0.99)   # Ngưỡng trên: phân vị 99%
        before_max = df_clean[col].max()     # Lưu max trước khi clip để so sánh
        df_clean[col] = df_clean[col].clip(lower=Q1, upper=Q99)  # Cắt giá trị về khoảng [Q1, Q99]
        after_max = df_clean[col].max()
        print(f"  • {col:25s}: [{Q1:6.2f}, {Q99:6.2f}]  "
              f"(max: {before_max:.2f} → {after_max:.2f})")

print(f"\n✅ Kích thước sau winsorization: {df_clean.shape} (số hàng không đổi!)")

## ⚙️ 5. Feature Engineering

Tạo **7 đặc trưng mới** dựa trên hiểu biết kinh doanh giúp mô hình học được các mẫu phức tạp hơn.

In [ ]:
df_feat = df_clean.copy()   # Tạo bản sao từ df đã qua winsorization

# ──── Đặc trưng 1: Is_New_Customer ────
# Khách hàng mới (Tenure < 3 tháng) có tỉ lệ churn cao nhất
df_feat['Is_New_Customer'] = (df_feat['Tenure'] < 3).astype(int)
# Tenure < 3 → 1 (khách mới); ngược lại → 0

# ──── Đặc trưng 2: Avg_Cashback_Per_Order ────
# Cashback trung bình mỗi đơn hàng – proxy cho "giá trị cảm nhận" của khách
df_feat['Avg_Cashback_Per_Order'] = (
    df_feat['CashbackAmount'] / (df_feat['OrderCount'].fillna(1) + 1)
    # +1 để tránh chia cho 0; fillna(1) cho khách chưa có đơn hàng
)

# ──── Đặc trưng 3: App_Engagement_Score ────
# Điểm gắn kết ứng dụng = số giờ dùng app × số thiết bị đăng ký
df_feat['App_Engagement_Score'] = (
    df_feat['HourSpendOnApp'] * df_feat['NumberOfDeviceRegistered']
)

# ──── Đặc trưng 4: Churn_Risk_Flag ────
# Khách mới + có khiếu nại = nhóm rủi ro cao nhất (interaction term)
df_feat['Churn_Risk_Flag'] = (
    (df_feat['Complain'] == 1) & (df_feat['Is_New_Customer'] == 1)
).astype(int)   # Cả hai điều kiện đúng → 1; ngược lại → 0

# ──── Đặc trưng 5: Long_Inactive ────
# Không đặt hàng lâu hơn median → sắp rời đi
median_days = df_feat['DaySinceLastOrder'].median()   # Tính median ngày không hoạt động
df_feat['Long_Inactive'] = (df_feat['DaySinceLastOrder'] > median_days).astype(int)

# ──── Đặc trưng 6: Dissatisfied_Complainant ────
# Hài lòng thấp (≤2) + có khiếu nại → nhóm nguy hiểm cực kỳ
df_feat['Dissatisfied_Complainant'] = (
    (df_feat['SatisfactionScore'] <= 2) & (df_feat['Complain'] == 1)
).astype(int)

# ──── Đặc trưng 7: Coupon_Dependency ────
# Tỉ lệ dùng coupon / tổng đơn – khách phụ thuộc khuyến mãi dễ rời khi hết
df_feat['Coupon_Dependency'] = (
    df_feat['CouponUsed'] / (df_feat['OrderCount'].fillna(1) + 1)
)

new_cols = ['Is_New_Customer', 'Avg_Cashback_Per_Order', 'App_Engagement_Score',
            'Churn_Risk_Flag', 'Long_Inactive', 'Dissatisfied_Complainant',
            'Coupon_Dependency']

print("✅ Đã tạo 7 đặc trưng mới:")
for col in new_cols:
    print(f"   • {col}")

print("\n📊 Thống kê mô tả đặc trưng mới:")
print(df_feat[new_cols].describe().round(3))

In [ ]:
# Vẽ tỉ lệ churn theo 4 đặc trưng nhị phân mới tạo
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
plot_cols = ['Is_New_Customer', 'Churn_Risk_Flag',
              'Long_Inactive', 'Dissatisfied_Complainant']

for i, col in enumerate(plot_cols):
    churn_r = df_feat.groupby(col)['Churn'].mean() * 100  # Tỉ lệ churn (%) theo giá trị 0/1
    churn_r.plot(kind='bar', ax=axes[i],
                  color=['#2196F3', '#F44336'],   # 0 = xanh, 1 = đỏ
                  edgecolor='black')
    axes[i].set_title(f'Tỉ lệ Churn theo {col}', fontweight='bold')
    axes[i].set_ylabel('Tỉ lệ Churn (%)')
    axes[i].tick_params(axis='x', rotation=0)
    for p in axes[i].patches:   # Ghi % lên đầu cột
        axes[i].annotate(f'{p.get_height():.1f}%',
                         (p.get_x() + p.get_width()/2., p.get_height()),
                         ha='center', va='bottom', fontweight='bold')

plt.suptitle('Tỉ lệ Churn theo Đặc trưng Mới', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔬 6. Thống kê Suy diễn (Inferential Statistics)

Kiểm định các **giả thuyết kinh doanh** bằng T-test (biến số) và Chi-square (biến phân loại).

### 6.1 T-test – Tenure giữa nhóm Churn và Không Churn

In [ ]:
print("=" * 65)
print("6.1 T-TEST: Tenure – Churn vs Không Churn")
print("=" * 65)
print("H0: Không có sự khác biệt về Tenure giữa hai nhóm")
print("H1: Khách churn có Tenure THẤP HƠN (khách mới dễ rời hơn)")
print()

tenure_churn   = df[df['Churn'] == 1]['Tenure'].dropna()   # Tenure của nhóm churn
tenure_noChurn = df[df['Churn'] == 0]['Tenure'].dropna()   # Tenure của nhóm không churn

# Welch's t-test (equal_var=False): không giả định phương sai bằng nhau
t_stat, p_val = ttest_ind(tenure_churn, tenure_noChurn, equal_var=False)

print(f"Nhóm Churn    – Tenure TB: {tenure_churn.mean():.2f} tháng (n={len(tenure_churn)})")
print(f"Nhóm Không Churn – Tenure TB: {tenure_noChurn.mean():.2f} tháng (n={len(tenure_noChurn)})")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value:     {p_val:.6f}")

# Kết luận dựa trên ngưỡng ý nghĩa α = 0.05
if p_val < 0.05:
    print("\n✅ Kết luận: BÁC BỎ H0 (p < 0.05)")
    print("   → Khách hàng mới (Tenure thấp) churn nhiều hơn đáng kể")
    print("   → Cần chương trình onboarding 30 ngày!")
else:
    print("\n❌ Không đủ bằng chứng bác bỏ H0")

# Vẽ boxplot và KDE để trực quan hoá
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(x='Churn', y='Tenure', data=df, ax=axes[0],
            palette=['#2196F3', '#F44336'])
axes[0].set_title(f'Tenure theo Churn (T={t_stat:.3f}, p={p_val:.4f})', fontweight='bold')
axes[0].set_xticklabels(['Không Churn', 'Churn'])

# KDE plot: so sánh phân phối mật độ xác suất của Tenure
sns.kdeplot(data=df[df['Churn']==0]['Tenure'].dropna(), ax=axes[1],
            label='Không Churn', fill=True, alpha=0.5, color='#2196F3')
sns.kdeplot(data=df[df['Churn']==1]['Tenure'].dropna(), ax=axes[1],
            label='Churn', fill=True, alpha=0.5, color='#F44336')
axes[1].set_title('Phân phối Tenure', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

### 6.2 T-test cho các đặc trưng số khác

In [ ]:
print("=" * 65)
print("6.2 T-TEST: Các đặc trưng số khác vs Churn")
print("=" * 65)

# Danh sách đặc trưng cần kiểm định kèm giả thuyết hướng
test_vars = [
    ('WarehouseToHome',  'Kho xa hơn → churn nhiều hơn'),
    ('CashbackAmount',   'Cashback thấp hơn → churn nhiều hơn'),
    ('SatisfactionScore','Hài lòng thấp hơn → churn nhiều hơn'),
    ('DaySinceLastOrder','Không hoạt động lâu hơn → churn nhiều hơn'),
    ('HourSpendOnApp',   'Dùng app ít hơn → churn nhiều hơn'),
]

ttest_results = []
for feat, hyp in test_vars:
    g1 = df[df['Churn'] == 1][feat].dropna()   # Nhóm churn
    g0 = df[df['Churn'] == 0][feat].dropna()   # Nhóm không churn
    t, p = ttest_ind(g1, g0, equal_var=False)  # Welch's t-test
    ttest_results.append({
        'Feature': feat,
        'Churn_Mean': round(g1.mean(), 3),       # Trung bình nhóm churn
        'NonChurn_Mean': round(g0.mean(), 3),    # Trung bình nhóm không churn
        'T-stat': round(t, 3),
        'p-value': round(p, 5),
        'Significant': '✅ CÓ' if p < 0.05 else '❌ KHÔNG',   # Có ý nghĩa thống kê?
        'Hypothesis': hyp
    })

ttest_df = pd.DataFrame(ttest_results)
print(ttest_df.to_string(index=False))

### 6.3 Kiểm định Chi-square – Biến phân loại vs Churn

In [ ]:
print("=" * 65)
print("6.3 KIỂM ĐỊNH CHI-SQUARE: Đặc trưng phân loại vs Churn")
print("=" * 65)

cat_test_cols = ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender',
                 'PreferedOrderCat', 'MaritalStatus', 'Complain', 'CityTier']

chi2_results = []
for col in cat_test_cols:
    ct = pd.crosstab(df[col], df['Churn'])        # Bảng chéo (contingency table)
    chi2, p, dof, _ = chi2_contingency(ct)        # Chi2, p-value, bậc tự do, tần suất kỳ vọng
    n = ct.sum().sum()                             # Tổng số quan sát
    cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))  # Cramér's V: cỡ hiệu ứng [0,1]
    effect = 'Mạnh' if cramers_v > 0.3 else ('Trung bình' if cramers_v > 0.1 else 'Yếu')
    chi2_results.append({
        'Feature': col,
        'Chi2': round(chi2, 3),
        'dof': dof,                               # Bậc tự do = (r-1)(c-1)
        'p-value': round(p, 5),
        "Cramér's V": round(cramers_v, 4),        # Càng gần 1 → liên hệ càng mạnh
        'Significant': '✅ CÓ' if p < 0.05 else '❌ KHÔNG',
        'Effect Size': effect
    })

chi2_df = pd.DataFrame(chi2_results).sort_values('Chi2', ascending=False)
print(chi2_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Biểu đồ tỉ lệ churn theo trạng thái khiếu nại (normalize='index' = theo hàng)
complain_churn = pd.crosstab(df['Complain'], df['Churn'], normalize='index') * 100
complain_churn.plot(kind='bar', ax=axes[0],
                     color=['#2196F3', '#F44336'], edgecolor='black', rot=0)
axes[0].set_title('Tỉ lệ Churn theo Khiếu nại', fontweight='bold')
axes[0].set_xlabel('Complain (0=Không, 1=Có)')
axes[0].set_ylabel('Phần trăm (%)')
axes[0].legend(['Không Churn', 'Churn'])

# Biểu đồ tỉ lệ churn theo CityTier
city_churn = df.groupby('CityTier')['Churn'].mean() * 100
city_churn.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black', rot=0)
axes[1].set_title('Tỉ lệ Churn (%) theo CityTier', fontweight='bold')
axes[1].set_ylabel('Tỉ lệ Churn (%)')
axes[1].set_xlabel('City Tier')
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                      (p.get_x() + p.get_width()/2., p.get_height()),
                      ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 💰 7. Hồi quy CLV – Customer Lifetime Value

**Định nghĩa CLV Proxy:** `CashbackAmount × log(1 + OrderCount)`
- Ý nghĩa: tổng giá trị cashback khách tạo ra, tương quan với tổng chi tiêu
- Dùng **Gradient Boosting Regressor** thay vì hồi quy tuyến tính

In [ ]:
print("=" * 60)
print("HỒI QUY CLV – GRADIENT BOOSTING REGRESSOR")
print("=" * 60)
print("Target: CLV_Proxy = CashbackAmount × log(1 + OrderCount)")
print()

# Bỏ hàng có missing ở 2 cột cần thiết để tính CLV
df_clv = df_feat.copy().dropna(subset=['CashbackAmount', 'OrderCount'])
df_clv['CLV_Proxy'] = df_clv['CashbackAmount'] * np.log1p(df_clv['OrderCount'])
# np.log1p(x) = log(1 + x) – tránh log(0) khi OrderCount = 0

# Đặc trưng đầu vào cho mô hình CLV
clv_features = ['Tenure', 'OrderCount', 'CouponUsed', 'DaySinceLastOrder',
                 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'SatisfactionScore',
                 'Is_New_Customer', 'App_Engagement_Score', 'Coupon_Dependency']

# Điền missing bằng KNN trước khi cho vào mô hình
knn_clv = KNNImputer(n_neighbors=5)
X_clv = pd.DataFrame(knn_clv.fit_transform(df_clv[clv_features]),
                      columns=clv_features)
y_clv = df_clv['CLV_Proxy'].values   # Biến mục tiêu (giá trị liên tục)

# Chia train/test 80/20
X_clv_tr, X_clv_te, y_clv_tr, y_clv_te = train_test_split(
    X_clv, y_clv, test_size=0.2, random_state=SEED)

# Khởi tạo và huấn luyện Gradient Boosting Regressor
gbr = GradientBoostingRegressor(
    n_estimators=200,       # 200 cây quyết định nối tiếp nhau
    learning_rate=0.05,     # Learning rate nhỏ → hội tụ chậm nhưng ổn định
    max_depth=4,            # Độ sâu tối đa mỗi cây (kiểm soát overfitting)
    random_state=SEED
)
gbr.fit(X_clv_tr, y_clv_tr)          # Huấn luyện trên tập train
y_clv_pred = gbr.predict(X_clv_te)   # Dự đoán trên tập test

r2   = r2_score(y_clv_te, y_clv_pred)                      # R² càng gần 1 càng tốt
rmse = np.sqrt(mean_squared_error(y_clv_te, y_clv_pred))   # RMSE: sai số dự đoán trung bình

print(f"📊 R²:   {r2:.4f}")
print(f"📊 RMSE: {rmse:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Thực tế vs Dự đoán (điểm lý tưởng nằm trên đường đỏ y=x)
axes[0].scatter(y_clv_te, y_clv_pred, alpha=0.4, color='steelblue')
axes[0].plot([y_clv_te.min(), y_clv_te.max()],
              [y_clv_te.min(), y_clv_te.max()], 'r--', lw=2)   # Đường tham chiếu y=x
axes[0].set_xlabel('CLV Thực tế')
axes[0].set_ylabel('CLV Dự đoán')
axes[0].set_title(f'Hồi quy CLV (GBR) – R²={r2:.4f}', fontweight='bold')

# Feature importance của Gradient Boosting
feat_imp_clv = pd.Series(gbr.feature_importances_,
                          index=clv_features).sort_values(ascending=True)
feat_imp_clv.plot(kind='barh', ax=axes[1], color='teal', edgecolor='black')
axes[1].set_title('Feature Importance – Hồi quy CLV', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Insight: CLV thấp + rủi ro churn cao = nhóm ưu tiên giữ chân cao nhất")

## 📐 8. SEM – Mô hình Phương trình Cấu trúc

Phân tích **quan hệ nhân quả**: Logistics & Chất lượng Dịch vụ → Hài lòng KH → Churn

**6 giả thuyết:**
| # | Giả thuyết | Dấu kỳ vọng |
|---|-----------|------------|
| H1 | WarehouseToHome → SatisfactionScore | − (kho xa → hài lòng thấp) |
| H2 | Complain → SatisfactionScore | − (khiếu nại → hài lòng thấp) |
| H3 | Tenure → SatisfactionScore | + (gắn bó lâu → hài lòng cao) |
| H4 | SatisfactionScore → Churn | − (hài lòng cao → ít churn) |
| H5 | Tenure → Churn | − (gắn bó lâu → ít churn) |
| H6 | Complain → Churn | + (khiếu nại → nhiều churn) |

In [ ]:
# Vẽ sơ đồ đường dẫn (path diagram) của SEM
fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis('off')   # Ẩn trục tọa độ

# Định nghĩa vị trí (x, y) của từng nút trong sơ đồ
nodes = {
    'WarehouseToHome':   (1.5, 5.5),   # Biến ngoại sinh (exogenous)
    'Complain':          (1.5, 3.5),   # Biến ngoại sinh
    'Tenure':            (1.5, 1.5),   # Biến ngoại sinh
    'SatisfactionScore': (5,   4.5),   # Biến trung gian (mediator)
    'Churn':             (8.5, 3),     # Biến kết quả (endogenous)
}

# Vẽ hình chữ nhật cho mỗi nút, tô màu theo loại biến
for name, (x, y) in nodes.items():
    color = ('#FFEBEE' if name == 'Churn' else            # Đỏ nhạt = biến kết quả
             '#FFF9C4' if name == 'SatisfactionScore' else # Vàng = biến trung gian
             '#E3F2FD')                                    # Xanh nhạt = biến ngoại sinh
    rect = plt.Rectangle((x-1.2, y-0.4), 2.4, 0.8, fill=True,
                          facecolor=color, edgecolor='navy', lw=2, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y, name, ha='center', va='center', fontsize=9,
             fontweight='bold', zorder=4)

# Hàm vẽ mũi tên có nhãn giả thuyết
def draw_arrow(start, end, label, color='navy'):
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    mx, my = (start[0]+end[0])/2, (start[1]+end[1])/2   # Điểm giữa để đặt nhãn
    ax.text(mx, my+0.15, label, ha='center', fontsize=9,
            color=color, fontweight='bold')

# Vẽ 6 đường dẫn theo giả thuyết (dấu − = đỏ, dấu + = xanh)
draw_arrow((2.7, 5.5), (3.8, 4.9), 'H1: β₁(-)', 'darkred')    # H1: WarehouseToHome → Satisfaction
draw_arrow((2.7, 3.5), (3.8, 4.5), 'H2: β₂(-)', 'darkred')    # H2: Complain → Satisfaction
draw_arrow((2.7, 1.5), (3.8, 4.1), 'H3: β₃(+)', 'darkgreen')  # H3: Tenure → Satisfaction
draw_arrow((6.2, 4.5), (7.3, 3.3), 'H4: β₄(-)', 'darkred')    # H4: Satisfaction → Churn
draw_arrow((2.7, 1.5), (7.3, 2.8), 'H5: β₅(-)', 'darkblue')   # H5: Tenure → Churn
draw_arrow((2.7, 3.5), (7.3, 3.0), 'H6: β₆(+)', 'darkred')    # H6: Complain → Churn

ax.set_title('Sơ đồ Đường dẫn SEM – Phân tích Churn\n'
              'Logistics & Chất lượng DV → Hài lòng → Churn',
              fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('SEM_path_diagram.png', dpi=150, bbox_inches='tight')   # Lưu file ảnh
plt.show()
print("✅ Đã lưu: SEM_path_diagram.png")

In [ ]:
# Fit mô hình SEM bằng semopy (nếu đã cài)
if SEM_AVAILABLE:
    print("🔬 Đang fit SEM với semopy...")

    sem_cols = ['WarehouseToHome', 'Complain', 'Tenure',
                'SatisfactionScore', 'Churn']
    df_sem = df[sem_cols].dropna()   # Bỏ hàng có NaN trước khi fit

    # Scale về [0,1] để các hệ số đường dẫn (β) có thể so sánh với nhau
    scaler_sem = MinMaxScaler()
    df_sem_scaled = pd.DataFrame(
        scaler_sem.fit_transform(df_sem), columns=sem_cols
    )

    # Định nghĩa cấu trúc mô hình bằng cú pháp R-style (~)
    sem_model_str = """
        SatisfactionScore ~ WarehouseToHome + Complain + Tenure
        Churn ~ SatisfactionScore + Tenure + Complain
    """
    model_sem = semopy.Model(sem_model_str)    # Khởi tạo mô hình SEM
    result_sem = model_sem.fit(df_sem_scaled)  # Tối ưu hoá tham số bằng Maximum Likelihood

    print("\n📊 Kết quả SEM (hệ số đường dẫn):")
    estimates = model_sem.inspect()   # Trả về bảng β, SE, t, p cho mỗi đường dẫn
    print(estimates.to_string(index=False))

    print("\n📈 Chỉ số độ khớp mô hình (model fit):")
    stats_sem = semopy.calc_stats(model_sem)   # CFI, RMSEA, χ², ...
    print(stats_sem.T)

    print("\n💡 Cách đọc kết quả:")
    print("   • Estimate: hệ số đường dẫn β")
    print("   • p-value < 0.05: đường dẫn có ý nghĩa thống kê")
    print("   • CFI > 0.90, RMSEA < 0.08: mô hình khớp tốt")
else:
    print("⚠️ semopy chưa cài – chỉ có sơ đồ đường dẫn")
    print("   Chạy: !pip install semopy rồi restart runtime")

## 🤖 9. Pipeline Tiền xử lý ML/DL

**Thứ tự:** One-Hot Encoding → KNN Imputer → Chia train/test → **SMOTE (chỉ train!)** → StandardScaler

⚠️ **QUAN TRỌNG:** SMOTE chỉ áp dụng trên TẬP TRAIN để tránh rò rỉ dữ liệu (data leakage)!

In [ ]:
print("=" * 60)
print("PIPELINE TIỀN XỬ LÝ ML")
print("=" * 60)

df_model = df_feat.copy().drop(columns=['CustomerID'])   # Bỏ cột ID không có giá trị dự đoán

# ─── Bước 1: One-Hot Encoding ───
# drop_first=True: bỏ cột đầu tiên của mỗi biến → tránh đa cộng tuyến (dummy variable trap)
cat_cols_ml = df_model.select_dtypes(include=['object']).columns.tolist()  # Tìm cột dạng chuỗi
print(f"Cột phân loại ({len(cat_cols_ml)}): {cat_cols_ml}")

df_model = pd.get_dummies(df_model, columns=cat_cols_ml, drop_first=True)  # Mã hoá nhị phân
print(f"Kích thước sau OHE: {df_model.shape}")

# ─── Bước 2: Tách X và y ───
X = df_model.drop(columns=['Churn']).astype(float)   # Tất cả cột trừ nhãn → float
y = df_model['Churn'].astype(int)                    # Nhãn nhị phân: 0 hoặc 1
print(f"\nSố đặc trưng: {X.shape[1]}")

# ─── Bước 3: KNN Imputer ───
print("\n🔄 Chạy KNN Imputer (k=5)...")
knn_imputer = KNNImputer(n_neighbors=5)   # Dùng 5 láng giềng gần nhất để điền missing
X_imputed = pd.DataFrame(knn_imputer.fit_transform(X), columns=X.columns)
print(f"Số missing còn lại: {X_imputed.isnull().sum().sum()}")   # Phải = 0

# ─── Bước 4: Chia train/test theo tỉ lệ nhãn (Stratified) ───
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y,
    test_size=0.2,        # 20% test, 80% train
    random_state=SEED,
    stratify=y            # Giữ nguyên tỉ lệ churn trong cả train và test
)
print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Tỉ lệ churn – train: {y_train.mean()*100:.2f}%")
print(f"Tỉ lệ churn – test:  {y_test.mean()*100:.2f}%")

# ─── Bước 5: SMOTE (CHỈ TRÊN TẬP TRAIN!) ───
print("\n🔄 Áp dụng SMOTE trên tập TRAIN...")
smote = SMOTE(random_state=SEED, k_neighbors=5)   # Tạo mẫu tổng hợp cho lớp thiểu số
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"Trước SMOTE: {dict(y_train.value_counts())}")
print(f"Sau SMOTE:   {dict(pd.Series(y_train_sm).value_counts())}")   # Hai lớp bằng nhau

# ─── Bước 6: StandardScaler (cho Deep Learning) ───
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)   # fit trên train → tính mean, std
X_test_scaled  = scaler.transform(X_test)           # transform trên test → dùng mean, std của train

print("\n✅ Tiền xử lý hoàn tất!")
print(f"   X_train (SMOTE + Scaled): {X_train_scaled.shape}")
print(f"   X_test  (Scaled):         {X_test_scaled.shape}")

### Hàm helper – Đánh giá mô hình

In [ ]:
# Danh sách lưu kết quả của tất cả mô hình để so sánh sau
results_list = []

def evaluate_model(name, y_true, y_pred, y_prob=None):
    """Tính toán và in đầy đủ các metrics cho một mô hình."""
    acc      = accuracy_score(y_true, y_pred)                         # Tỉ lệ dự đoán đúng
    f1_w     = f1_score(y_true, y_pred, average='weighted')           # F1 có trọng số theo lớp
    f1_churn = f1_score(y_true, y_pred, pos_label=1)                  # F1 riêng cho lớp Churn (quan trọng nhất)
    prec     = precision_score(y_true, y_pred, pos_label=1, zero_division=0)  # Precision lớp Churn
    rec      = recall_score(y_true, y_pred, pos_label=1, zero_division=0)     # Recall lớp Churn
    auc_s    = roc_auc_score(y_true, y_prob) if y_prob is not None else None  # AUC-ROC (cần xác suất)
    pr_auc   = average_precision_score(y_true, y_prob) if y_prob is not None else None  # PR-AUC

    print(f"\n{'='*55}")
    print(f"  MODEL: {name}")
    print(f"{'='*55}")
    print(f"  Accuracy:               {acc:.4f}")
    print(f"  F1 (weighted):          {f1_w:.4f}")
    print(f"  F1 (Churn=1):           {f1_churn:.4f}")   # Metric quan trọng nhất
    print(f"  Precision (Churn):      {prec:.4f}")
    print(f"  Recall (Churn):         {rec:.4f}")
    if auc_s:  print(f"  ROC-AUC:                {auc_s:.4f}")
    if pr_auc: print(f"  PR-AUC:                 {pr_auc:.4f}")
    print(f"\n  Báo cáo phân loại:")
    print(classification_report(y_true, y_pred,
                                 target_names=['Không Churn', 'Churn']))

    return {'Model': name, 'Accuracy': acc, 'F1_weighted': f1_w,
            'F1_Churn': f1_churn, 'Precision': prec, 'Recall': rec,
            'ROC_AUC': auc_s, 'PR_AUC': pr_auc}


def plot_confusion_roc(name, y_true, y_pred, y_prob=None):
    """Vẽ Ma trận Nhầm lẫn và Đường cong ROC cạnh nhau."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Ma trận nhầm lẫn: hàng = thực tế, cột = dự đoán
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Không Churn', 'Churn'],
                yticklabels=['Không Churn', 'Churn'])
    axes[0].set_title(f'{name} – Ma trận Nhầm lẫn', fontweight='bold')
    axes[0].set_ylabel('Thực tế')
    axes[0].set_xlabel('Dự đoán')

    if y_prob is not None:
        fpr, tpr, _ = roc_curve(y_true, y_prob)   # False/True Positive Rate theo ngưỡng
        roc_auc = auc(fpr, tpr)                    # Diện tích dưới đường ROC
        axes[1].plot(fpr, tpr, lw=2.5, label=f'AUC = {roc_auc:.4f}')
        axes[1].plot([0, 1], [0, 1], 'k--', lw=1)   # Đường ngẫu nhiên (random classifier)
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].set_title(f'{name} – Đường cong ROC', fontweight='bold')
        axes[1].legend()

    plt.tight_layout()
    plt.show()

print("✅ Hàm helper đã sẵn sàng")

## ⚡ 10. ML Model 1 – LightGBM + SHAP + 5-Fold CV

> **Tại sao LightGBM?** Mô hình Gradient Boosting mạnh nhất trên dữ liệu bảng mất cân bằng (benchmark Kaggle 2022-2024). Hỗ trợ SHAP natively.

In [ ]:
print("🚀 Huấn luyện LightGBM + 5-Fold Stratified CV...")

lgb_model = LGBMClassifier(
    n_estimators=500,          # Số cây (vòng boosting)
    learning_rate=0.05,        # Tốc độ học nhỏ → tránh overfitting
    max_depth=6,               # Độ sâu tối đa mỗi cây
    num_leaves=40,             # Số lá tối đa (kiểm soát độ phức tạp)
    min_child_samples=30,      # Số mẫu tối thiểu ở lá → regularization
    subsample=0.8,             # 80% hàng ngẫu nhiên mỗi cây → giảm variance
    colsample_bytree=0.8,      # 80% cột ngẫu nhiên mỗi cây → giảm overfitting
    reg_alpha=0.1,             # L1 regularization
    reg_lambda=0.1,            # L2 regularization
    class_weight='balanced',   # Tự động tăng trọng số lớp thiểu số (Churn)
    random_state=SEED,
    verbose=-1                 # Tắt log huấn luyện
)

# ─── 5-Fold Stratified Cross-Validation ───
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_auc = cross_val_score(lgb_model, X_imputed, y,
                          cv=skf,
                          scoring='roc_auc',   # Metric đánh giá: AUC-ROC
                          n_jobs=-1)           # Dùng tất cả CPU để chạy song song
print(f"\n📊 5-Fold CV ROC-AUC:")
for i, s in enumerate(cv_auc, 1):
    print(f"   Fold {i}: {s:.4f}")
print(f"   Trung bình: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")

# ─── Huấn luyện trên tập SMOTE-balanced ───
lgb_model.fit(X_train_sm, y_train_sm)   # Fit với dữ liệu sau SMOTE

y_pred_lgb = lgb_model.predict(X_test)              # Dự đoán nhãn (0/1)
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]  # Xác suất thuộc lớp Churn (cột index 1)

result_lgb = evaluate_model('LightGBM', y_test, y_pred_lgb, y_prob_lgb)
result_lgb['CV_AUC_mean'] = cv_auc.mean()   # Thêm CV AUC vào kết quả
results_list.append(result_lgb)
plot_confusion_roc('LightGBM', y_test, y_pred_lgb, y_prob_lgb)

In [ ]:
# ─── Giải thích mô hình bằng SHAP ───
print("📊 Tính SHAP values cho LightGBM...")

explainer   = shap.TreeExplainer(lgb_model)     # TreeExplainer tối ưu cho tree-based models
shap_values = explainer.shap_values(X_test)     # Tính SHAP cho toàn bộ tập test

# Một số phiên bản SHAP trả về list [class0_vals, class1_vals] cho bài toán nhị phân
if isinstance(shap_values, list):
    shap_vals = shap_values[1]   # Lấy SHAP của lớp Churn (index 1)
else:
    shap_vals = shap_values      # Một số phiên bản trả về trực tiếp

print("\n🔍 SHAP Summary Plot – Top 20 đặc trưng quan trọng nhất")
# Điểm đỏ = SHAP cao (đẩy về Churn), điểm xanh = SHAP thấp (đẩy về Không Churn)
shap.summary_plot(shap_vals, X_test, max_display=20, show=True)

print("\n📊 SHAP Bar Plot – Feature Importance tổng hợp")
# |SHAP| trung bình của mỗi đặc trưng trên toàn bộ dataset
shap.summary_plot(shap_vals, X_test, plot_type='bar',
                   max_display=15, show=True)

print("\n🔎 SHAP Dependence Plot – Tenure (đặc trưng quan trọng nhất)")
# Xem SHAP của Tenure thay đổi như thế nào theo giá trị Tenure
shap.dependence_plot('Tenure', shap_vals, X_test, show=True)

## 🐱 11. ML Model 2 – CatBoost

> **Tại sao CatBoost?** Xử lý tốt đặc trưng phân loại không cần label encoding. Xếp hạng #1 trên nhiều Kaggle Fraud/Churn benchmark 2023-2024.

In [ ]:
print("🚀 Huấn luyện CatBoost...")
print("Tài liệu: Prokhorenkova et al., NeurIPS 2018")
print()

cat_model = CatBoostClassifier(
    iterations=500,                  # Số vòng boosting (tương đương n_estimators)
    learning_rate=0.05,              # Tốc độ học
    depth=6,                         # Độ sâu cây (CatBoost dùng oblivious trees)
    l2_leaf_reg=3,                   # Regularization L2 tại nút lá
    border_count=128,                # Số ngưỡng phân chia cho đặc trưng số
    auto_class_weights='Balanced',   # Tự cân bằng trọng số lớp (xử lý mất cân bằng)
    random_seed=SEED,
    verbose=0                        # Tắt log
)

# ─── 5-Fold CV trên tập đã SMOTE ───
cv_auc_cat = cross_val_score(
    cat_model, X_train_sm, y_train_sm,
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring='roc_auc', n_jobs=-1
)
print(f"📊 5-Fold CV ROC-AUC: {cv_auc_cat.mean():.4f} ± {cv_auc_cat.std():.4f}")

# ─── Huấn luyện với eval_set để theo dõi hiệu suất trên test ───
cat_model.fit(
    X_train_sm, y_train_sm,
    eval_set=(X_test, y_test),   # Theo dõi loss trên tập test sau mỗi 100 vòng
    verbose=100
)

y_pred_cat = cat_model.predict(X_test).flatten().astype(int)   # flatten() đảm bảo 1D array
y_prob_cat = cat_model.predict_proba(X_test)[:, 1]

result_cat = evaluate_model('CatBoost', y_test, y_pred_cat, y_prob_cat)
result_cat['CV_AUC_mean'] = cv_auc_cat.mean()
results_list.append(result_cat)
plot_confusion_roc('CatBoost', y_test, y_pred_cat, y_prob_cat)

# Feature importance của CatBoost (dựa trên mức giảm loss khi dùng đặc trưng đó)
cat_feat_imp = pd.Series(
    cat_model.get_feature_importance(), index=X_test.columns
).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
cat_feat_imp.sort_values().plot(kind='barh', color='coral', edgecolor='black')
plt.title('CatBoost – Top 15 Feature Importance', fontweight='bold')
plt.xlabel('Điểm Importance')
plt.tight_layout()
plt.show()

## 🤖 12. DL Model 1 – FT-Transformer (Feature Tokenizer Transformer)

> **Tại sao FT-Transformer?** Gorishniy et al. (NeurIPS 2021) chứng minh FT-Transformer vượt MLP và ResNet trên phân loại dữ liệu bảng.

**Kiến trúc:**
```
Features → Feature Tokenizer → [CLS] + Tokens → Transformer Encoder → MLP Head → Dự đoán
```

In [ ]:
# ─── Kiến trúc FT-Transformer ───
# Tài liệu: Gorishniy et al., NeurIPS 2021. Paper: https://arxiv.org/abs/2106.11959

class FeatureTokenizer(nn.Module):
    """Chiếu mỗi đặc trưng vào không gian embedding d_token chiều."""
    def __init__(self, n_features, d_token):
        super().__init__()
        # Ma trận trọng số riêng cho mỗi đặc trưng: shape (n_features, d_token)
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        # Bias riêng cho mỗi đặc trưng: shape (n_features, d_token)
        self.bias   = nn.Parameter(torch.empty(n_features, d_token))
        nn.init.kaiming_uniform_(self.weight)   # Khởi tạo He Uniform (tốt cho ReLU-like)
        nn.init.zeros_(self.bias)               # Khởi tạo bias = 0

    def forward(self, x):
        # x: (batch, n_features) → (batch, n_features, 1) * weight + bias
        return x.unsqueeze(-1) * self.weight + self.bias
        # Kết quả: (batch, n_features, d_token)


class FTTransformer(nn.Module):
    """
    Feature Tokenizer + Transformer cho dữ liệu bảng.
    Tài liệu: Gorishniy et al., NeurIPS 2021.
    """
    def __init__(self, n_features, d_token=64, n_heads=8, n_layers=3,
                 d_ffn=256, dropout=0.1, n_classes=1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)   # Bước tokenize

        # Token [CLS] học được – giống BERT, đại diện cho toàn chuỗi
        self.cls_token = nn.Parameter(torch.empty(1, 1, d_token))
        nn.init.normal_(self.cls_token, std=0.02)   # Khởi tạo Gaussian nhỏ

        # Một tầng Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token,           # Kích thước embedding
            nhead=n_heads,             # Số đầu attention
            dim_feedforward=d_ffn,     # Kích thước lớp FFN ẩn
            dropout=dropout,           # Xác suất dropout
            batch_first=True,          # Input shape: (batch, seq, d_model)
            norm_first=True            # Pre-LayerNorm: ổn định hơn Post-LN khi training
        )
        # Xếp chồng n_layers tầng encoder
        self.transformer = nn.TransformerEncoder(encoder_layer,
                                                  num_layers=n_layers)

        # MLP head để phân loại từ representation của [CLS]
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),           # Chuẩn hoá trước khi phân loại
            nn.Linear(d_token, d_token // 2),# Giảm chiều
            nn.GELU(),                        # Hàm kích hoạt GELU (mượt hơn ReLU)
            nn.Dropout(dropout),
            nn.Linear(d_token // 2, n_classes)  # Đầu ra: 1 logit (binary)
        )

    def forward(self, x):
        tokens  = self.tokenizer(x)                        # (batch, n_feat, d_token)
        cls     = self.cls_token.expand(x.size(0), -1, -1) # (batch, 1, d_token) – broadcast
        tokens  = torch.cat([cls, tokens], dim=1)           # (batch, n_feat+1, d_token)
        out     = self.transformer(tokens)                  # Transformer xử lý chuỗi tokens
        cls_out = out[:, 0, :]                              # Lấy output của token [CLS] (vị trí 0)
        return self.head(cls_out).squeeze(-1)               # (batch,) – squeeze chiều cuối

print("✅ Kiến trúc FT-Transformer đã định nghĩa")

In [ ]:
# ─── Huấn luyện FT-Transformer ───
print("🚀 Huấn luyện FT-Transformer...")

N_FEATURES = X_train_scaled.shape[1]   # Số đặc trưng sau tiền xử lý

# ─── Chuyển numpy array sang PyTorch Tensor ───
X_tr_t = torch.FloatTensor(X_train_scaled).to(DEVICE)   # Train features lên GPU/CPU
y_tr_t = torch.FloatTensor(y_train_sm.values).to(DEVICE) # Train labels
X_te_t = torch.FloatTensor(X_test_scaled).to(DEVICE)    # Test features

# ─── DataLoader: chia batch và shuffle khi train ───
train_ds     = TensorDataset(X_tr_t, y_tr_t)               # Gộp X và y vào dataset
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)  # Batch 256, shuffle mỗi epoch

# ─── Khởi tạo mô hình ───
ft_model = FTTransformer(
    n_features=N_FEATURES,
    d_token=64,    # Kích thước embedding mỗi token
    n_heads=8,     # 8 đầu attention (Multi-Head Attention)
    n_layers=3,    # 3 tầng Transformer encoder
    d_ffn=256,     # Kích thước FFN bên trong mỗi encoder
    dropout=0.1    # 10% dropout
).to(DEVICE)

n_params = sum(p.numel() for p in ft_model.parameters())   # Đếm tổng số tham số
print(f"Tổng tham số: {n_params:,}")

# ─── Optimizer, Scheduler, Loss ───
optimizer_ft = optim.AdamW(ft_model.parameters(), lr=1e-3, weight_decay=1e-4)
# AdamW = Adam + weight decay thực sự (L2 reg đúng cách)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=50)
# Cosine Annealing: lr giảm dần theo hàm cosine → tăng chất lượng hội tụ
criterion_ft = nn.BCEWithLogitsLoss()
# BCEWithLogitsLoss = sigmoid + BCE tích hợp → ổn định số học hơn BCE(sigmoid(x))

# ─── Vòng lặp huấn luyện với Early Stopping ───
N_EPOCHS       = 60    # Số epoch tối đa
PATIENCE       = 10    # Dừng sớm nếu val AUC không cải thiện sau 10 epoch liên tiếp
best_val_auc   = 0     # Theo dõi AUC tốt nhất
best_state     = None  # Lưu trọng số tốt nhất
train_losses, val_aucs = [], []   # Lưu lịch sử để vẽ đồ thị
patience_counter = 0

for epoch in range(N_EPOCHS):
    ft_model.train()   # Chế độ training (bật dropout, BatchNorm running stats)
    ep_loss = 0
    for xb, yb in train_loader:
        optimizer_ft.zero_grad()              # Xoá gradient cũ
        logits = ft_model(xb)                 # Forward pass: (batch,)
        loss   = criterion_ft(logits, yb)     # Tính loss
        loss.backward()                       # Backpropagation: tính gradient
        nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)  # Cắt gradient nếu > 1.0 (tránh exploding)
        optimizer_ft.step()                   # Cập nhật tham số
        ep_loss += loss.item()
    scheduler_ft.step()   # Cập nhật learning rate theo lịch

    # ─── Validation sau mỗi epoch ───
    ft_model.eval()   # Chế độ evaluation (tắt dropout)
    with torch.no_grad():   # Không tính gradient → tiết kiệm bộ nhớ
        val_logits = ft_model(X_te_t).cpu().numpy()
        val_probs  = 1 / (1 + np.exp(-val_logits))   # Sigmoid thủ công: logit → xác suất
        val_auc    = roc_auc_score(y_test.values, val_probs)

    train_losses.append(ep_loss / len(train_loader))   # Loss TB mỗi epoch
    val_aucs.append(val_auc)

    # ─── Early Stopping logic ───
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state   = {k: v.clone() for k, v in ft_model.state_dict().items()}  # Deep copy
        patience_counter = 0   # Reset đếm
    else:
        patience_counter += 1  # Tăng đếm nếu không cải thiện

    if patience_counter >= PATIENCE:
        print(f"⏹️  Dừng sớm ở epoch {epoch+1}")
        break

    if (epoch + 1) % 10 == 0:   # In log mỗi 10 epoch
        print(f"   Epoch {epoch+1:3d} | Loss: {train_losses[-1]:.4f} | Val AUC: {val_auc:.4f}")

# ─── Khôi phục trọng số tốt nhất ───
ft_model.load_state_dict(best_state)
print(f"\n✅ Val AUC tốt nhất: {best_val_auc:.4f}")

# ─── Dự đoán cuối cùng ───
ft_model.eval()
with torch.no_grad():
    final_logits = ft_model(X_te_t).cpu().numpy()
    y_prob_ft    = 1 / (1 + np.exp(-final_logits))   # Sigmoid
    y_pred_ft    = (y_prob_ft >= 0.5).astype(int)    # Ngưỡng 0.5 → nhãn nhị phân

result_ft = evaluate_model('FT-Transformer', y_test, y_pred_ft, y_prob_ft)
results_list.append(result_ft)
plot_confusion_roc('FT-Transformer', y_test, y_pred_ft, y_prob_ft)

# ─── Vẽ đường cong học ───
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(train_losses, color='steelblue')
axes[0].set_title('FT-Transformer – Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')

axes[1].plot(val_aucs, color='darkorange')
axes[1].axhline(best_val_auc, ls='--', color='red',
                 label=f'AUC tốt nhất = {best_val_auc:.4f}')
axes[1].set_title('FT-Transformer – Validation AUC', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROC-AUC')
axes[1].legend()

plt.tight_layout()
plt.show()

## 🌟 13. DL Model 2 – TabNet (dựa trên Attention)

> **Tại sao TabNet?** Arik & Pfister (AAAI 2021) – chọn đặc trưng tuần tự bằng attention. Vừa deep learning, vừa có thể giải thích.

In [ ]:
print("🚀 Huấn luyện TabNet...")
print("Tài liệu: Arik & Pfister, AAAI 2021")
print()

# ─── Chuyển về numpy float32 (TabNet yêu cầu) ───
X_train_tn = X_train_sm.values.astype(np.float32)   # Giá trị float32 cho train
y_train_tn = y_train_sm.values.astype(int)           # Nhãn integer
X_test_tn  = X_test.values.astype(np.float32)        # Test features (chưa scale – TabNet tự scale)
y_test_tn  = y_test.values.astype(int)

tabnet_model = TabNetClassifier(
    n_d=32,              # Chiều của representation D (decision step)
    n_a=32,              # Chiều của attention embedding A
    n_steps=5,           # Số bước attention tuần tự (sequential)
    gamma=1.3,           # Hệ số relaxation cho attention (>1 = cho phép dùng lại đặc trưng)
    n_independent=2,     # Số lớp độc lập trong mỗi step
    n_shared=2,          # Số lớp chia sẻ giữa các step
    lambda_sparse=1e-4,  # Hệ số regularization sparsity (khuyến khích chọn ít đặc trưng)
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={'step_size': 10, 'gamma': 0.9},  # Giảm lr 10% mỗi 10 epoch
    mask_type='entmax',  # entmax-15: phân phối attention thưa hơn softmax
    verbose=10,          # In log mỗi 10 epoch
    seed=SEED
)

tabnet_model.fit(
    X_train_tn, y_train_tn,
    eval_set=[(X_test_tn, y_test_tn)],  # Theo dõi AUC trên test
    eval_metric=['auc'],                 # Metric theo dõi
    max_epochs=100,                      # Tối đa 100 epoch
    patience=15,                         # Dừng sớm sau 15 epoch không cải thiện
    batch_size=512,                      # Kích thước batch
    virtual_batch_size=128,              # Ghost Batch Normalization (TabNet đặc trưng)
    num_workers=0,                       # Số worker đọc data (0 = main thread)
    drop_last=False                      # Giữ batch cuối dù không đủ kích thước
)

y_pred_tn = tabnet_model.predict(X_test_tn)              # Dự đoán nhãn
y_prob_tn = tabnet_model.predict_proba(X_test_tn)[:, 1]  # Xác suất lớp Churn

result_tn = evaluate_model('TabNet', y_test_tn, y_pred_tn, y_prob_tn)
results_list.append(result_tn)
plot_confusion_roc('TabNet', y_test_tn, y_pred_tn, y_prob_tn)

# ─── Feature Importance từ trọng số attention ───
feat_imp_tn = pd.Series(
    tabnet_model.feature_importances_,  # Ma trận attention trung bình qua các step
    index=X_test.columns
).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
feat_imp_tn.sort_values().plot(kind='barh', color='mediumpurple', edgecolor='black')
plt.title('TabNet – Top 15 Feature Importance (Attention)', fontweight='bold')
plt.xlabel('Điểm Importance')
plt.tight_layout()
plt.show()

## 📊 14. So sánh Mô hình – Tổng hợp

In [ ]:
results_df = pd.DataFrame(results_list)   # Tạo DataFrame từ danh sách kết quả 4 mô hình
results_df = results_df.sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
results_df.index += 1   # Index bắt đầu từ 1 thay vì 0

print("=" * 85)
print("SO SÁNH MÔ HÌNH – STAT3013 CHURN PREDICTION")
print("=" * 85)
display_cols = ['Model', 'Accuracy', 'F1_weighted', 'F1_Churn',
                'Precision', 'Recall', 'ROC_AUC', 'PR_AUC']
print(results_df[display_cols].to_string(index=True))

# In mô hình tốt nhất dựa trên ROC-AUC
print(f"\n🏆 Mô hình tốt nhất (ROC-AUC): {results_df.iloc[0]['Model']}")
print(f"   ROC-AUC: {results_df.iloc[0]['ROC_AUC']:.4f}")
print(f"   F1-Churn: {results_df.iloc[0]['F1_Churn']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))   # 3 biểu đồ: F1, ROC-AUC, PR-AUC

metrics = ['F1_Churn', 'ROC_AUC', 'PR_AUC']
titles  = ['F1-Score (Lớp Churn)', 'Điểm ROC-AUC', 'Điểm PR-AUC']
colors  = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']   # Mỗi màu cho một mô hình

for i, (metric, title) in enumerate(zip(metrics, titles)):
    bars = axes[i].bar(results_df['Model'], results_df[metric],
                        color=colors[:len(results_df)], edgecolor='black')
    axes[i].set_title(title, fontweight='bold', fontsize=13)
    axes[i].set_ylim(0.5, 1.05)   # Trục y từ 0.5 để dễ so sánh
    axes[i].tick_params(axis='x', rotation=20)

    # Tô viền vàng cho cột tốt nhất
    best_idx = results_df[metric].idxmax() - 1
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

    for bar, val in zip(bars, results_df[metric]):
        axes[i].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                      f'{val:.4f}', ha='center', va='bottom',
                      fontsize=9, fontweight='bold')

plt.suptitle('🏆 So sánh Mô hình – Customer Churn Prediction',
              fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')   # Lưu ảnh
plt.show()
print("✅ Đã lưu: model_comparison.png")

In [ ]:
# ─── Vẽ ROC và PR curves của tất cả 4 mô hình trên cùng một đồ thị ───
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

model_probs = [
    ('LightGBM',       y_prob_lgb, '#2196F3'),   # (tên, xác suất dự đoán, màu)
    ('CatBoost',       y_prob_cat, '#4CAF50'),
    ('FT-Transformer', y_prob_ft,  '#FF9800'),
    ('TabNet',         y_prob_tn,  '#9C27B0'),
]

y_test_arr = np.array(y_test)   # Chuyển nhãn thực tế sang numpy array

# ─── Đồ thị ROC ───
for name, probs, color in model_probs:
    fpr, tpr, _ = roc_curve(y_test_arr, probs)  # Tính FPR, TPR theo từng ngưỡng
    roc_auc_val = auc(fpr, tpr)                  # Diện tích dưới đường ROC
    axes[0].plot(fpr, tpr, lw=2.5,
                  label=f'{name} (AUC={roc_auc_val:.4f})', color=color)

axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Ngẫu nhiên (AUC=0.5)')  # Đường baseline
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('Đường cong ROC – Tất cả Mô hình', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.4)

# ─── Đồ thị Precision-Recall ───
for name, probs, color in model_probs:
    prec_c, rec_c, _ = precision_recall_curve(y_test_arr, probs)  # Tính theo ngưỡng
    pr_auc_val = auc(rec_c, prec_c)
    axes[1].plot(rec_c, prec_c, lw=2.5,
                  label=f'{name} (PR-AUC={pr_auc_val:.4f})', color=color)

baseline_pr = y_test_arr.mean()   # Tỉ lệ Churn = baseline của PR
axes[1].axhline(baseline_pr, ls='--', color='k', lw=1.5,
                 label=f'Ngẫu nhiên (PR={baseline_pr:.3f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Đường cong Precision-Recall (tập trung mất cân bằng)',
                   fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.4)

plt.suptitle('Đường cong Đánh giá – Tất cả Mô hình', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Đã lưu: roc_pr_curves.png")

## 📤 15. Xuất CSV cho Dashboard Looker Studio

**Quy trình:**
1. Xuất CSV bên dưới
2. Upload lên Google Sheets
3. Mở Looker Studio → tạo nguồn dữ liệu từ Google Sheets
4. Xây dựng 3 trang: **Executive Summary | Chẩn đoán | Danh sách Hành động**

(Xem file `HUONG_DAN_LOOKER_STUDIO.md` để biết chi tiết.)

In [ ]:
# ─── Chọn mô hình tốt nhất để xuất ───
# Mặc định: LightGBM (ổn định + có SHAP). Đổi sang mô hình khác nếu cần.
best_model_name = 'LightGBM'
best_probs = y_prob_lgb   # Xác suất churn từ LightGBM
best_preds = y_pred_lgb   # Nhãn dự đoán từ LightGBM

# Ghi chú: Để dùng CatBoost thay thế, uncomment 3 dòng dưới:
# best_model_name = 'CatBoost'
# best_probs = y_prob_cat
# best_preds = y_pred_cat

# ─── Tạo DataFrame export từ dữ liệu gốc của tập test ───
test_indices = y_test.index           # Lấy chỉ số hàng của tập test trong df gốc
df_export = df.loc[test_indices].copy()  # Giữ thông tin khách hàng gốc (không scale)

# Thêm các cột dự đoán
df_export['Churn_Probability'] = np.round(best_probs, 4)  # Xác suất churn (4 chữ số)
df_export['Predicted_Churn']   = best_preds                # Nhãn dự đoán (0/1)

# ─── Phân mức rủi ro theo xác suất churn ───
df_export['Risk_Level'] = pd.cut(
    best_probs,
    bins=[0, 0.3, 0.6, 0.8, 1.01],                            # Ngưỡng phân vùng
    labels=['Thấp', 'Trung bình', 'Cao', 'Rất cao']           # Nhãn mức độ rủi ro
)

# ─── Lý do churn hàng đầu từ SHAP ───
if isinstance(shap_values, list):
    sv = shap_values[1]   # Lớp Churn
else:
    sv = shap_values

# Tạo DataFrame SHAP với index khớp với tập test
shap_df = pd.DataFrame(sv, columns=X_test.columns, index=test_indices)
# Tìm đặc trưng có |SHAP| lớn nhất → lý do chính khiến khách hàng có khả năng churn
df_export['Top_Churn_Reason'] = shap_df.abs().idxmax(axis=1).values

# ─── Hành động khuyến nghị dựa trên logic kinh doanh ───
def suggest_action(row):
    if row['Complain'] == 1 and row['Churn_Probability'] > 0.6:
        return 'Khẩn: Gửi voucher 50k trong 1 giờ'         # Khách khiếu nại + rủi ro cao
    elif row['Tenure'] < 3 and row['Churn_Probability'] > 0.4:
        return 'Onboarding: Miễn ship 3 đơn tiếp theo'     # Khách mới có nguy cơ
    elif row['SatisfactionScore'] <= 2:
        return 'Khảo sát + gọi điện cá nhân'               # Hài lòng thấp
    elif row['Churn_Probability'] > 0.8:
        return 'VIP: Gọi điện & ưu đãi độc quyền'          # Rủi ro rất cao
    else:
        return 'Theo dõi: Đưa vào chương trình loyalty'    # Nhóm bình thường

df_export['Recommended_Action'] = df_export.apply(suggest_action, axis=1)
df_export['Model_Used'] = best_model_name   # Ghi nhận mô hình dự đoán

# ─── Chọn cột xuất ───
export_cols = [
    'CustomerID', 'Churn', 'Predicted_Churn', 'Churn_Probability',
    'Risk_Level', 'Top_Churn_Reason', 'Recommended_Action', 'Model_Used',
    'Tenure', 'SatisfactionScore', 'Complain',
    'WarehouseToHome', 'CityTier', 'PreferedOrderCat',
    'CashbackAmount', 'OrderCount', 'DaySinceLastOrder',
    'Gender', 'MaritalStatus', 'PreferredPaymentMode'
]
export_cols = [c for c in export_cols if c in df_export.columns]  # Chỉ lấy cột tồn tại

# ─── Lưu file CSV ───
df_export[export_cols].to_csv('churn_predictions_dashboard.csv', index=False,
                               encoding='utf-8-sig')   # utf-8-sig để Excel đọc được tiếng Việt

print("✅ Đã lưu: churn_predictions_dashboard.csv")
print(f"   • Số bản ghi:             {len(df_export):,}")
print(f"   • Rủi ro Cao + Rất cao:   "
      f"{(df_export['Risk_Level'].isin(['Cao', 'Rất cao'])).sum()}")
print(f"\n📋 Xem trước (10 hàng):")
df_export[export_cols].head(10)

In [ ]:
# ─── Thống kê tóm tắt cho dashboard ───
print("=" * 60)
print("THỐNG KÊ TÓM TẮT DASHBOARD")
print("=" * 60)

total_customers      = len(df_export)
high_risk            = (df_export['Risk_Level'].isin(['Cao', 'Rất cao'])).sum()
avg_cashback_at_risk = df_export[
    df_export['Risk_Level'].isin(['Cao', 'Rất cao'])
]['CashbackAmount'].mean()
est_revenue_loss = high_risk * avg_cashback_at_risk * 12  # Ước tính thiệt hại doanh thu năm

print(f"Tổng khách hàng (tập test):      {total_customers:,}")
print(f"Rủi ro Cao + Rất cao:            {high_risk:,} "
      f"({high_risk/total_customers*100:.1f}%)")
print(f"Cashback TB nhóm rủi ro:         {avg_cashback_at_risk:,.2f}")
print(f"Ước tính doanh thu có nguy cơ/năm: {est_revenue_loss:,.0f}")

print("\nTỉ lệ churn dự đoán theo CityTier:")
by_city = df_export.groupby('CityTier').agg(
    Tổng_KH=('CustomerID', 'count'),
    Tỉ_lệ_Churn_DĐ=('Predicted_Churn', 'mean')
)
by_city['Tỉ_lệ_Churn_DĐ'] = (by_city['Tỉ_lệ_Churn_DĐ'] * 100).round(2)
print(by_city)

print("\n🎯 Bước tiếp theo:")
print("   1. Tải file churn_predictions_dashboard.csv")
print("   2. Upload lên Google Sheets (drive.google.com)")
print("   3. Mở Looker Studio (lookerstudio.google.com)")
print("   4. Kết nối nguồn dữ liệu → Google Sheets")
print("   5. Xây dựng 3 trang: Executive / Chẩn đoán / Hành động")
print("\n📖 Xem HUONG_DAN_LOOKER_STUDIO.md để biết hướng dẫn chi tiết")

## 🏁 16. Kết luận & Ý nghĩa Kinh doanh

In [ ]:
print("=" * 75)
print("  KẾT LUẬN & Ý NGHĨA KINH DOANH – DỰ ĐOÁN CHURN KHÁCH HÀNG")
print("=" * 75)

print("""
📊 TÓM TẮT DỮ LIỆU & PHƯƠNG PHÁP:
────────────────────────────────────────────────────────────
 • Dataset:         5.630 khách hàng | 20 đặc trưng | 16.84% churn
 • Mất cân bằng:    SMOTE oversampling (chỉ tập train)
 • Tiền xử lý:      KNN Imputer + Winsorization + OHE(drop_first=True)
 • Feature Eng:     7 đặc trưng mới có ý nghĩa kinh doanh
 • Metrics:         F1, ROC-AUC, PR-AUC (KHÔNG chỉ dùng Accuracy)
 • Validation:      5-Fold Stratified Cross-Validation

🤖 MÔ HÌNH (SOTA 2024-2025):
────────────────────────────────────────────────────────────
 ML1: LightGBM         – Gradient Boosting nhanh, hỗ trợ SHAP
 ML2: CatBoost         – Xử lý phân loại, NeurIPS 2018
 DL1: FT-Transformer   – Transformer dữ liệu bảng, NeurIPS 2021
 DL2: TabNet           – DL có attention, AAAI 2021
""")

print("🏆 KẾT QUẢ:")
for idx, row in results_df.iterrows():
    medal = "🥇" if idx == 1 else ("🥈" if idx == 2 else ("🥉" if idx == 3 else "  "))
    print(f"  {medal} {row['Model']:20s}: "
           f"AUC={row['ROC_AUC']:.4f} | F1={row['F1_Churn']:.4f} | "
           f"PR-AUC={row['PR_AUC']:.4f}")

print("""

💡 NGUYÊN NHÂN CHURN HÀNG ĐẦU (từ SHAP – LightGBM):
────────────────────────────────────────────────────────────
 1. Complain = 1           → Khách đã khiếu nại, rủi ro rất cao
 2. Tenure thấp            → Khách mới (0-3 tháng) churn nhiều nhất
 3. WarehouseToHome xa     → Giao hàng chậm → thất vọng
 4. SatisfactionScore thấp → Trải nghiệm kém
 5. CashbackAmount thấp    → Cảm thấy không được trân trọng

🎯 KHUYẾN NGHỊ KINH DOANH (có thể thực hiện ngay):
────────────────────────────────────────────────────────────
 ✅ Hỗ trợ:      Complain=1 → tự động gửi voucher 50k trong 1 giờ
 ✅ Onboarding:  Chương trình 30 ngày: miễn ship + cashback bậc thang
 ✅ Logistics:   Mở kho nhỏ tại Tier 2-3, tối ưu SLA giao hàng
 ✅ VIP:         Gọi điện khách được đánh dấu 'Rủi ro rất cao'
 ✅ Voucher thông minh: Chỉ dành cho nhóm 'Rủi ro cao nhưng thuyết phục được'
 ✅ Ưu tiên CLV: Ngân sách giữ chân → CLV cao + rủi ro churn cao
""")

print("=" * 75)
print("✅ Pipeline hoàn chỉnh | STAT3013 – Dự đoán Churn Khách hàng")
print("=" * 75)